# compare DTW vs Siamese coverage

In [ ]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().parent
PROC = project_root / "data" / "processed"

pairs = pd.read_parquet(PROC / "pairs_dev.parquet")
dtw   = pd.read_parquet(PROC / "dtw_cache_dev.parquet")
siam  = pd.read_parquet(PROC / "siam_cache_dev.parquet")

print("pairs:", len(pairs), " dtw:", len(dtw), " siam:", len(siam))
missing_in_dtw  = set(pairs.pair_id) - set(dtw.pair_id)
missing_in_siam = set(pairs.pair_id) - set(siam.pair_id)
print("missing in DTW:", len(missing_in_dtw), " | missing in Siamese:", len(missing_in_siam))


# Compare ROC summaries across scenarios (Siamese)

In [ ]:
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

PROC = Path.cwd().parent / "data" / "processed"
metrics_dir = PROC / "metrics"
summary_path = metrics_dir / "siamese_summary.json"
summary = pd.read_json(summary_path)
display(summary[["scenario","auc_dev","eer_dev","auc_test","eer_test","apcer_test","bpcer_test"]])

# Simple side-by-side bars for TEST AUC/EER
fig, ax = plt.subplots(figsize=(7,4))
x = np.arange(len(summary))
ax.bar(x - 0.15, summary["auc_test"], width=0.3, label="AUC (test)")
ax.bar(x + 0.15, summary["eer_test"], width=0.3, label="EER (test)")
ax.set_xticks(x); ax.set_xticklabels(summary["scenario"], rotation=0)
ax.set_ylim(0, 1.0); ax.set_ylabel("Score"); ax.set_title("Siamese — TEST AUC/EER by scenario")
ax.legend(); fig.tight_layout()
plt.show()


# DTW vs Siamese (scatter on DEV)

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

PROC = Path.cwd().parent / "data" / "processed"
dev_pairs = pd.read_parquet(PROC / "pairs_dev.parquet")
dev_dtw   = pd.read_parquet(PROC / "dtw_cache_dev.parquet")
dev_siam  = pd.read_parquet(PROC / "siam_cache_dev.parquet")

df = dev_pairs.merge(dev_dtw, on="pair_id").merge(dev_siam, on="pair_id")
# DTW similarity = -d_mean; Siamese similarity = siam_mean
df["dtw_sim"] = -df["d_mean"]

fig, ax = plt.subplots(figsize=(6,5))
ax.scatter(df["dtw_sim"], df["siam_mean"], s=6, alpha=0.3)
ax.set_xlabel("DTW similarity (−d_mean)"); ax.set_ylabel("Siamese probability (mean)")
ax.set_title("DEV: DTW vs Siamese (per pair)")
plt.tight_layout(); plt.show()